# MRI Pattern Detection Demo

This is an educational computer vision starter project.

It uses small synthetic grayscale images to simulate an image-classification workflow:
- create images
- train a classifier
- let the student choose a sample image
- show prediction and confidence

This is **not** a real medical system and not for diagnosis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import ipywidgets as widgets
from IPython.display import display, clear_output
np.random.seed(42)

In [ ]:
def make_image(bright_spot=False, size=12):
    img = np.random.normal(0.35, 0.08, (size, size))
    img = np.clip(img, 0, 1)
    if bright_spot:
        img[4:8,4:8] += 0.35
    return np.clip(img, 0, 1)

images = []
labels = []
for _ in range(50):
    images.append(make_image(False)); labels.append(0)
for _ in range(50):
    images.append(make_image(True)); labels.append(1)

images = np.array(images)
labels = np.array(labels)

X = images.reshape(len(images), -1)
y = labels
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.25)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
print("Model trained.")
print("Test accuracy:", round(model.score(X_test, y_test), 3))

In [ ]:
sample_idx = widgets.IntSlider(description="Sample", min=0, max=len(X_test)-1, value=0)
button = widgets.Button(description="Analyze Image")
out = widgets.Output()

def analyze(_):
    with out:
        clear_output()
        sample = X_test[sample_idx.value].reshape(1, -1)
        pred = int(model.predict(sample)[0])
        probs = model.predict_proba(sample)[0]

        fig, ax = plt.subplots(1, 2, figsize=(8,3))
        ax[0].imshow(sample.reshape(12,12), cmap="gray")
        ax[0].set_title("Selected Image")
        ax[0].axis("off")
        ax[1].bar(["Class 0","Class 1"], probs)
        ax[1].set_ylim(0,1)
        ax[1].set_title(f"Prediction: {'Pattern Found' if pred==1 else 'No Pattern'}")
        plt.show()

button.on_click(analyze)
display(sample_idx, button, out)